#  OpenMP for GPU


Enter your name and student ID.

 * Name:
 * Student ID:


# 1. Introduction
* <a href="http://openmp.org/" target="_blank" rel="noopener">OpenMP</a> is the de facto programming model for multicore environment
* More recently, it supports GPU offloading
* In this notebook you are going to learn OpenMP for GPU
* Consult [the spec](https://www.openmp.org/spec-html/5.2/openmp.html) when necessary
* Take a look at [a talk slide OPENMP IN NVIDIA'S HPC by Jeff Larkin](https://openmpcon.org/wp-content/uploads/openmpcon2021-nvidia.pdf)

# 2. How to work on this notebook
* Read text, modify cells if necessary, and execute them
* Many cells can be executed without modifying them
* Cells that need your work are typically preceded with <font color="green">text colored green explaining what you need to do</font>
* Problems with non-trivial work have a section header starting with <font color="green">"Problem X :"</font>

# 3. Jupyter AI
* You can ask Jupyter AI when you need help
* Open [pd05_omp_gpu.chat](pd05_omp_gpu.chat) in this folder
* Choose `OpenCode` in Persona Chooser
* You can try several models, but `LiteLLM (taulec)/GPT-5.3 Codex (UTokyo Azure)` is recommended for now
* It is instructed to help you but not to give you the answer directly to exercise problems; do not try to circumvent it

# 4. Setting up compilers
- The setup is the same as for the OpenMP we used previously for the CPU


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH

## 4-1. Summary of compiler options to compile OpenMP programs for GPU
* `nvc`/`nvc++` : `-mp=gpu` option
* `clang`/`clang++` : `-fopenmp --offload-arch=sm_90` options
  * `sm_90` is the _compute capability_ of the GH200 GPU on Miyabi (9.0); on another machine you put that machine's number there

# 5. Summary of directives you are going to learn
* [`#pragma omp target`](https://www.openmp.org/spec-html/5.2/openmpse85.html#x215-22500013.8) : offloads the immediately following statement to the device
* [`#pragma omp teams`](https://www.openmp.org/spec-html/5.2/openmpse58.html#x170-17600010.2) : creates a number of teams (similar to `#pragma omp parallel`)
* [`#pragma omp distribute`](https://www.openmp.org/spec-html/5.2/openmpse67.html#x189-19500011.6) : distributes iterations of the immediately following for loop to teams
* [`#pragma omp parallel`](https://www.openmp.org/spec-html/5.2/openmpse57.html#x165-17100010.1) : creates a number of threads within a team
* [`#pragma omp for`](https://www.openmp.org/spec-html/5.2/openmpse66.html#x185-19100011.5) : distributes iterations of the immediately following for loop to threads of a team
* [`#pragma omp target data`](https://www.openmp.org/spec-html/5.2/openmpse82.html#x212-22100013.5)


# 6. [`#pragma omp target`](https://www.openmp.org/spec-html/5.2/openmpse85.html#x215-22500013.8) $\sim$ moving work to a GPU
* <font color="blue">syntax</font>
```
#pragma omp target
    S
```
executes $S$ on (_offloads_ $S$ to) a device (typically a GPU), with just one CUDA thread



In [ ]:
%%writefile omp_target.cc
#include <stdio.h>
int main() {
  printf("hello on host\n");
#pragma omp target
  printf("hello from target (hopefully GPU)\n");
  printf("back on host\n");
  return 0;
}

* Compiling


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -fopenmp --offload-arch=sm_90 omp_target.cc -o omp_target
# nvc++ -mp=gpu omp_target.cc -o omp_target

* Running


In [ ]:
%%sub
./omp_target

note

* while using `target` almost always intends to use a GPU, it can actually run without a GPU (fallback)
* executing the above program results in an identical result whether your machine has a GPU or not
  

In [ ]:
%%bash
./omp_target

* while good for portability, it may be confusing, so you can force it to run on GPU or signal an error when GPU is not available, by setting environment variable `OMP_TARGET_OFFLOAD=MANDATORY`.


In [ ]:
%%sub
# force it to run on GPU or signal an error
OMP_TARGET_OFFLOAD=MANDATORY ./omp_target

* the following _should_ fail, because it is run on a login node, which has no GPU


In [ ]:
%%bash --no
OMP_TARGET_OFFLOAD=MANDATORY ./omp_target

* `OMP_TARGET_OFFLOAD=DISABLED` has the opposite effect; it does not use GPU even if it is available

# 7. [`#pragma omp teams`](https://www.openmp.org/spec-html/5.2/openmpse58.html#x170-17600010.2)
## 7-1. basics
* <font color="blue">syntax</font>
```
#pragma omp target
#pragma omp teams
    S
```
creates a number of _teams_ and the master of each team will execute $S$

* Note:
  * `teams` must appear right inside `target` --- the spec forbids _any_ statement in the `target` region outside the `teams` construct
  * as such, `target` and `teams` are almost always used in the combined form (`#pragma omp target teams`), like this
```
#pragma omp target teams
    S
```

* `#pragma omp teams` it is similar to `#pragma omp parallel` in the sense that the effect is to have many threads execute the same statement
* `teams` is an extra layer of parallelism _outside_ `parallel` (`parallel` is a construct that creates threads _within_ a team)
* in implementation terms, you can think of a team a thread block of CUDA (we confirm it later)


In [ ]:
%%writefile omp_teams.cc
#include <stdio.h>
int main() {
  printf("hello on host\n");
#pragma omp target
#pragma omp teams
  printf("hello, I am the master of a team\n");
  printf("back on host\n");
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall -fopenmp --offload-arch=sm_90 omp_teams.cc -o omp_teams
# nvc++ -mp=gpu omp_teams.cc -o omp_teams

* Running
  * Specify the number of teams by `OMP_NUM_TEAMS` environment variable


In [ ]:
%%sub
OMP_TARGET_OFFLOAD=MANDATORY OMP_NUM_TEAMS=3 ./omp_teams

* Note: the behavior when omitting `OMP_NUM_TEAMS` is implementation-dependent and it seems
  * `clang++` : 3200
  * `nvc++` : the available SM count, which is 132 on a whole node (`lecture` queue) and 32 or 26 on a MIG instance (`lecture-mig` queue)

* Note: <font color=red>there is no way for a program to ask what the default will be _before_ the kernel runs</font>; we checked
  * `omp_get_max_teams()` returns 0 on the host with both compilers (0 means "I have not been told; the implementation will decide")
  * `omp_set_num_teams(7)` changes that host value to 7, but the kernel still launched 3200 teams under `clang++` --- just like `OMP_NUM_THREADS`, it never reaches the device
  * calling `omp_get_max_teams()` inside a `target` region does not even link under `clang++` (`nvlink error : Undefined reference to 'omp_get_max_teams'`) and aborts at run time under `nvc++` (`Fatal error: NOT IMPLEMENTED`)
  * `omp_get_num_procs()` inside a `target` region returns 2048 under `clang++` (not the SM count) and is likewise not implemented by `nvc++`
* so the only reliable ways are to _ask for_ a number yourself (`num_teams(n)` or `OMP_NUM_TEAMS`), or to look at `omp_get_num_teams()` from inside the region after the fact

## 7-2. getting team ID and the number of teams
* just as `omp_get_thread_num()` and `omp_get_num_threads()` tell you the thread ID and the number of threads of your team, you can get the team ID and the number of teams by
  * `omp_get_num_teams()` 
  * `omp_get_team_num()`


In [ ]:
%%writefile omp_team_num.cc
#include <stdio.h>
#include <omp.h>
int main() {
  printf("hello on host\n");
#pragma omp target
#pragma omp teams
  printf("in teams: %03d/%03d\n", omp_get_team_num(), omp_get_num_teams());
  printf("back on host\n");
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall -fopenmp --offload-arch=sm_90 omp_team_num.cc -o omp_team_num
# nvc++ -mp=gpu omp_team_num.cc -o omp_team_num

* Running


In [ ]:
%%sub
OMP_TARGET_OFFLOAD=MANDATORY OMP_NUM_TEAMS=5 ./omp_team_num

# 8. [`#pragma omp distribute`](https://www.openmp.org/spec-html/5.2/openmpse67.html#x189-19500011.6) $\sim$ distributing iterations to teams
* <font color="blue">syntax</font>
```
#pragma omp target
#pragma omp teams
    {
      ...
#pragma omp distribute
      for (...) {
        ...
      }
    }
```
distributes iterations of the for-loop across teams

* that is, <font color=blue>`distribute` is to `teams` what `for` is to `parallel`</font>


In [ ]:
%%writefile omp_distribute.cc
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>
int main(int argc, char ** argv) {
  int i = 1;
  int m = (argc > i ? atoi(argv[i]) : 5); i++;
  printf("hello on host\n");
#pragma omp target
#pragma omp teams
  {
    printf("in teams: %03d/%03d\n", omp_get_team_num(), omp_get_num_teams());
#pragma omp distribute
    for (int i = 0; i < m; i++) {
      printf("in distribute: i=%03d executed by %03d/%03d\n",
             i, omp_get_team_num(), omp_get_num_teams());
    }
  }
  printf("back on host\n");
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall -fopenmp --offload-arch=sm_90 omp_distribute.cc -o omp_distribute
# nvc++ -mp=gpu omp_distribute.cc -o omp_distribute

In [ ]:
%%sub
OMP_TARGET_OFFLOAD=MANDATORY OMP_NUM_TEAMS=3 ./omp_distribute 5

* note: if there is no statements between `teams` and `distribute` they can be combined into one directive


In [ ]:
%%writefile omp_target_teams_distribute.cc
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>
int main(int argc, char ** argv) {
  int i = 1;
  int m = (argc > i ? atoi(argv[i]) : 5);
  printf("hello on host\n");
#pragma omp target teams distribute
  for (int i = 0; i < m; i++) {
    printf("in distribute: i=%03d executed by %03d/%03d\n",
           i, omp_get_team_num(), omp_get_num_teams());
  }
  printf("back on host\n");
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall -fopenmp --offload-arch=sm_90 omp_target_teams_distribute.cc -o omp_target_teams_distribute
# nvc++ -mp=gpu omp_target_teams_distribute.cc -o omp_target_teams_distribute

In [ ]:
%%sub
OMP_TARGET_OFFLOAD=MANDATORY OMP_NUM_TEAMS=3 ./omp_target_teams_distribute 7

* note:
  * with only `teams` (and `distribute`), you have a number of thread blocks each having only _one_ thread, which is hardly useful
  * to effectively use GPUs, you need to use `parallel` within each team

# 9. [`#pragma omp parallel`](https://www.openmp.org/spec-html/5.2/openmpse57.html#x165-17100010.1)
## 9-1. `parallel` inside `teams`
* syntax:
```
#pragma omp target
#pragma omp teams
    {
      ...
#pragma omp parallel
      S
    }
```
creates a number of threads within each team

* recall that you used `parallel` to create threads when executing on CPUs
* used inside `teams`, it will create threads within the team, each executing $S$

* here is an example that illustrates it

<font color="blue"><tt>OMP_NUM_TEAMS=$T$ OMP_NUM_THREADS=$H$ ./omp_parallel</tt></font> creates $T$ teams each of which create $H$ threads


In [ ]:
%%writefile omp_parallel.cc
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>
int getenv_int(const char * v) {
  char * s = getenv(v);
  if (!s) {
    fprintf(stderr, "specify environment variable %s\n", v);
    exit(1);
  }
  return atoi(s);
}

int main(int argc, char ** argv) {
  int n_threads= getenv_int("OMP_NUM_THREADS");
  printf("hello on host\n");
#pragma omp target teams
  {
    printf("in teams: %03d/%03d\n", omp_get_team_num(), omp_get_num_teams());
#pragma omp parallel num_threads(n_threads)
    printf("in parallel: %03d/%03d %03d/%03d\n",
           omp_get_team_num(), omp_get_num_teams(),
           omp_get_thread_num(), omp_get_num_threads());
  }
  printf("back on host\n");
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall -fopenmp --offload-arch=sm_90 omp_parallel.cc -o omp_parallel
# nvc++ -mp=gpu omp_parallel.cc -o omp_parallel

In [ ]:
%%sub
OMP_TARGET_OFFLOAD=MANDATORY OMP_NUM_TEAMS=3 OMP_NUM_THREADS=32 ./omp_parallel

## 9-2. Note: `OMP_NUM_THREADS` environment variable does not reach the GPU
* on a CPU you can set the number of threads created by `parallel` either with the `OMP_NUM_THREADS` environment variable or with a `num_threads(x)` clause
* on a GPU, <font color=red>`OMP_NUM_THREADS` has no effect</font>; you must use `num_threads(x)` in your program
* The sample program above reads the environment variable _themselves_ (`getenv_int("OMP_NUM_THREADS")`) and pass the value to a `num_threads(...)` clause for convenience

## 9-3. Note: <font color=red>You do not always get the threads you asked for</font>
* `num_threads(H)` is a _request_, not an order
* We will examine it later and make sense of it, but for now, it suffices to understand they are subject to block size limitations of CUDA (e.g., due to register requirements and hardware limit (1024), whichever is smaller)
* what you _can_ rely on is that `omp_get_num_threads()`, called inside the `parallel` region, tells you the number you actually got (we confirmed it agrees with the number of threads that really ran)

## 9-4. Note: what you get when you omit `num_threads`
* if you write a bare `#pragma omp parallel` (no `num_threads` clause) inside `teams`, the default thread block size is used (128 for `clang++`, 1024 for `nvc++`), and `omp_get_num_threads()` reports
  * `clang++` : **128** when `parallel` is immediately inside `target`, **96** otherwise
  * `nvc++` : **992** in both cases
* the odd-looking 96 and 992 are the block size minus 32; a warp is reserved for the code _outside_ `parallel` regions
* we will see where the default block sizes come from later in [Peeking underneath]
* this is one more reason to say what you want with `num_threads(...)` rather than leave it to the implementation


# 10. [`#pragma omp for`](https://www.openmp.org/spec-html/5.2/openmpse66.html#x185-19100011.5)
* syntax:
```
#pragma omp target
#pragma omp teams
    ...
#pragma omp distribute
    for (...) {
#pragma omp parallel
      {
        ...
#pragma omp for
        for (...) {
            ...
        }
      }
    }
```

* used inside `parallel`, it will distribute iterations of the loop to threads



In [ ]:
%%writefile omp_for.cc
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>
int getenv_int(const char * v) {
  char * s = getenv(v);
  if (!s) {
    fprintf(stderr, "specify environment variable %s\n", v);
    exit(1);
  }
  return atoi(s);
}

int main(int argc, char ** argv) {
  int n_threads = getenv_int("OMP_NUM_THREADS");
  int i = 1;
  int m = (argc > i ? atoi(argv[i]) : 5); i++;
  int n = (argc > i ? atoi(argv[i]) : 6); i++;
  printf("hello on host\n");
#pragma omp target teams
  {
    printf("in teams: %03d/%03d\n", omp_get_team_num(), omp_get_num_teams());
#pragma omp distribute
    for (int i = 0; i < m; i++) {
      printf("in distribute: i=%03d executed by %03d/%03d\n",
             i, omp_get_team_num(), omp_get_num_teams());
#pragma omp parallel num_threads(n_threads)
      {
        printf("in parallel: i=%03d %03d/%03d %03d/%03d\n",
               i, omp_get_team_num(), omp_get_num_teams(),
               omp_get_thread_num(), omp_get_num_threads());
#pragma omp for
        for (int j = 0; j < n; j++) {
          printf("in for: i=%03d j=%03d executed by %03d/%03d %03d/%03d\n",
                 i, j, omp_get_team_num(), omp_get_num_teams(),
                 omp_get_thread_num(), omp_get_num_threads());
        }
      }
    }
  }
  printf("back on host\n");
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall -fopenmp --offload-arch=sm_90 omp_for.cc -o omp_for
# nvc++ -mp=gpu omp_for.cc -o omp_for

In [ ]:
%%sub
OMP_TARGET_OFFLOAD=MANDATORY OMP_NUM_TEAMS=3 OMP_NUM_THREADS=32 ./omp_for 5 6

# 11. Common combined directives
* anybody who has a right mind will feel sick with the whole series of different directive names that have little or no consistency
* each of them is nominally an independent, standalone directive, but many of them are almost always used together in practice
* since the purpose is often to execute a loop nest in parallel, most typically they are used in one of the following forms

* (most common) combine everything 
```
#pragma omp target teams distribute parallel for
    for (...) {
      S
    }
```

* parallelize an outer loop with `teams` $+$ `distribute` and an inner loop with `parallel` $+$ `for`

```
#pragma omp target teams distribute
    for (...) {
#pragma omp parallel for
      for (...) {
        ...
      }
    }  
```

# 12. [`#pragma omp target data`](https://www.openmp.org/spec-html/5.2/openmpse82.html#x212-22100013.5)
* `#pragma omp target data` clause specifies which variables/arrays are shared or transferred between CPU and GPU when the `target` region starts and finishes

## 12-1. Basics
* Thanks to the shared memory between CPU and GPU (NVLink-C2C), data sharing has been much simplified in OpenMP, too
* All data are automatically shared between CPU and GPU, except _scalar_ (not array) locals or globals are _copied_ from CPU to GPU when the `target` region starts, but _not copied back from GPU to CPU_ when it finishes
* That is, the following program just works, giving the result that most people consider natural
* For portability's sake, we have the following at the beginning
```
#pragma omp requires unified_shared_memory
```
which says the program relies on shared memory between CPU and GPU
* `nvc++` needs `-gpu=mem:unified` compile-time option to compile programs using that pragma


In [ ]:
%%writefile omp_data.cc
#include <assert.h>
#include <stdio.h>
#include <stdlib.h>
#include <sys/mman.h>
#include <err.h>

#pragma omp requires unified_shared_memory

void print_array(long * p, long n) {
  printf("{");
  for (int i = 0; i < n; i++) {
    if (i > 0) printf(", ");
    printf("%ld", p[i]);
  }
  printf("}\n");
}

// global array
#define MAX_N 100
long g[MAX_N];

int main(int argc, char ** argv) {
  int n               = (argc > 1 ? atoi(argv[1]) : 10);
  assert(n <= MAX_N);
  // local array
  long l[MAX_N];
  // heap (new)
  long * h = new long[n];
  // heap (mmap)
  long * m = (long *)mmap(NULL, n * sizeof(long), PROT_READ | PROT_WRITE, MAP_ANONYMOUS | MAP_PRIVATE, -1, 0);
  if (m == MAP_FAILED) err(1, "mmap");
#pragma omp target teams distribute parallel for
  for (int i = 0; i < n; i++) {
    g[i] = i * i;
  }
#pragma omp target teams distribute parallel for
  for (int i = 0; i < n; i++) {
    l[i] = i * i;
  }
#pragma omp target teams distribute parallel for
  for (int i = 0; i < n; i++) {
    h[i] = i * i;
  }
#pragma omp target teams distribute parallel for
  for (int i = 0; i < n; i++) {
    m[i] = i * i;
  }
  print_array(g, n);
  print_array(l, n);
  print_array(h, n);
  print_array(m, n);
  delete[] h;
  if (munmap(m, n * sizeof(long)) == -1) err(1, "munmap");
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall -fopenmp --offload-arch=sm_90 omp_data.cc -o omp_data
# nvc++ -mp=gpu -Minfo=mp omp_data.cc -o omp_data

In [ ]:
%%sub
OMP_TARGET_OFFLOAD=MANDATORY ./omp_data

## 12-2. Scalar local/global variables
* One thing that might be different from what people might expect is that modifications to scalar variables by GPU do not propagate to CPU
* In a way, it is similar to the behavior of passing scalar values to a function; a function gets a _copy_ of the value, and assigning to the formal variable does not change the value of the caller
* To override this behavior, add `map(tofrom: variable)`


In [ ]:
%%writefile omp_data_scalar.cc
/*** Scalar variables are the exception to "everything is shared":
     they are passed to the GPU by value (like a function argument),
     so assignments made on the GPU do not come back to the CPU.
     This holds for globals and locals alike.
     map(tofrom: ...) (or defaultmap) overrides it.                   ***/
#include <stdio.h>

#pragma omp requires unified_shared_memory

int g = 1;                      /* a global scalar */

int main() {
  int  l    = 1;                /* a local scalar  */
  long a[2] = { 1, 1 };         /* an array, for comparison */

  /* no map clause : both scalars are implicitly firstprivate */
#pragma omp target
  { g = 9; l = 9; a[0] = 9; }
  printf("no map clause             : g = %d, l = %d, a[0] = %ld\n", g, l, a[0]);

  g = 1; l = 1; a[0] = 1;
  /* map(tofrom: ...) : copied in *and* copied back */
#pragma omp target map(tofrom: g, l)
  { g = 9; l = 9; a[0] = 9; }
  printf("map(tofrom: g, l)         : g = %d, l = %d, a[0] = %ld\n", g, l, a[0]);

  g = 1; l = 1; a[0] = 1;
  /* the same, for every scalar in the region */
#pragma omp target defaultmap(tofrom: scalar)
  { g = 9; l = 9; a[0] = 9; }
  printf("defaultmap(tofrom: scalar): g = %d, l = %d, a[0] = %ld\n", g, l, a[0]);

  /* the CPU -> GPU direction always happens, with or without a map clause */
  g = 7; l = 7;
  int gg = -1, ll = -1;
#pragma omp target map(from: gg, ll)
  { gg = g; ll = l; }
  printf("values the GPU sees       : g = %d, l = %d\n", gg, ll);

  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall -fopenmp --offload-arch=sm_90 omp_data_scalar.cc -o omp_data_scalar
# nvc++ -mp=gpu -Minfo=mp omp_data_scalar.cc -o omp_data_scalar

In [ ]:
%%sub
OMP_TARGET_OFFLOAD=MANDATORY ./omp_data_scalar

## 12-3. A more general account on `map` clause
* `map` clause is designed to easily move/synchronize data between GPU and CPU when data are not shared between them by hardware
* as far as correctness is concerned, this has largely become unnecessary on hardware supporting CPU-GPU shared memory, except `map(tofrom: x)` is necessary to get the value of scalar variables back to CPU after `target` finishes
* when it comes to performance, what matters is on which processor (CPU or GPU) the data _physically_ resides, which is obscure anyway even when you use these map clauses
* therefore, mastering `map` clause is almost unnecessary when hardware support CPU-GPU shared memory, but I will nevertheless give a brief description below to give you a better overview

* <font color="blue">syntax:</font>
```
#pragma omp target data map(to: ...) map(from: ...) map(tofrom: ...) ...
    S
```
where ... is a variable, array name, or base address + range (e.g., a[0:n])

* basically, these clauses say that specified variables, arrays, or address ranges are valid expressions you can get "expected" values in the during or after $S$
* more specifically, 
  * those specified in `map(to: ...)` become valid on GPU during $S$
  * those specified in `map(from: ...)` become valid on CPU after $S$
* to accomplish that, the <font color="blue">_mapping_</font> between CPU address and GPU address are maintained by the runtime system and contents may be moved to or from GPU as necessary
  * data specified in `map(to: ...)` may be copied to GPU (CPU -&gt; GPU) before $S$
  * data specified in `map(from: ...)` may be copied from GPU (GPU -&gt; CPU) after $S$
* `map(tofrom: ...)` has the effect of both; it makes data available to GPU during $S$ and to CPU after $S$
* you can change the default for all scalars with `defaultmap(tofrom: scalar)`

* think it has two effects
  * one is "transfer data" that may be accessed from GPU
  * the other is "redirecting pointers" so that the same expression (e.g., a, a[i], p->x) accesses different locations depending on whether you are on GPU or CPU

* on a GPU with its own separate memory:

 | allocated as/by       | syntax                         | CPU -> GPU       | GPU -> CPU              |
 |-----------------------|--------------------------------|------------------|-------------------------|
 | local/global variable | `int v;`                       |                  | `map(from:v)` (\*)      |
 | local/global array    | `int a[N];`                    |                  | `map(from:a[p:q])` (\*) |
 | malloc/new            | `int * h = (int *)malloc(..);` | `map(to:h[p:q])` | `map(from:h[p:q])`      |
 |                       | `int * h = new int[N];`        | `map(to:h[p:q])` | `map(from:h[p:q])`      |
 | mmap                  | `int * h = (int *)mmap(..);`   | `map(to:h[p:q])` | `map(from:h[p:q])`      |

* on a GPU supporting CPU-GPU shared memory or with `#pragma omp requires unified_shared_memory`:

 | allocated as/by       | syntax                         | CPU -> GPU | GPU -> CPU         |
 |-----------------------|--------------------------------|------------|--------------------|
 | local/global variable | `int v;`                       |            | `map(from:v)` (\*) |
 |                       |                                |            |                    |
 | local/global array    | `int a[N];`                    |            |                    |
 | malloc/new            | `int * h = (int *)malloc(..);` |            |                    |
 |                       | `int * h = new int[N];`        |            |                    |
 | mmap                  | `int * h = (int *)mmap(..);`   |            |                    |

(\*) it should be `map(tofrom:v)` when GPU needs to read data written by CPU before the kernel launches

# 13. Synchronization Primitives
* they are the same constructs you already used on CPU, but not all of them survive the trip to a GPU

 |construct   |across teams|within a team|
 |------------|------------|-------------|
 |`atomic`    |OK          |OK           |
 |`reduction` |OK          |OK           |
 |`barrier`   |no          |OK           |
 |`critical`  |<font color=red>do not use</font>|<font color=red>do not use</font>|

* `barrier` binds to the innermost enclosing `parallel` region, so it never synchronizes different teams, however few teams you asked for; writing it directly inside `teams` is an error.  <font color=blue>If you need teams to synchronize, split the work into two `target` regions</font> --- the end of a `target` region is the device-wide barrier
* `critical` is not usable on a GPU: `nvc++` rejects it (`omp critical is not supported in GPU region`) and `clang++` accepts it without a warning but does not actually enforce mutual exclusion, so you silently get wrong answers.  Use `atomic` instead, which is also far faster

# 14. Peeking underneath
* With the two-level hierarchy, you might have already guessed that a team in OpenMP is actually a thread block in CUDA
* While the spec does not say anything about it, it indeed seems the case, as you will see in the following experiment
* Then we might expect the number of threads you specified with `num_threads(H)` becomes the thread block size, but as already said, it is not as simple as that
* The fundamental constraint is that a thread block size must be specified at the moment of the kernel launch, which happens at `#pragma omp target` in OpenMP
* On the other hand, the number of threads is specified with `#pragma omp parallel`, which may not be known at the time of executing `#pragma omp target`
* There are also cases where `#pragma omp parallel` are executed multiple times within a single `#pragma omp target` region, potentially with different number of threads
* The thread block size chosen by the system in those cases is not necessarily a good number
* Understanding it is therefore worth for performance

## 14-1. Showing kernel launch parameters and counting the number of threads
* you can show info about kernel launches and their launch parameters
  * `clang++` : run with `LIBOMPTARGET_INFO=-1`
  * `nvc++` : run with `NVCOMPILER_ACC_NOTIFY=1`
  * `nvc++` will also tell you at compile time, with `-Minfo=mp`
  
* the following programs count the actual number of launched threads by having each of them atomically increment the same variable
* we used two versions, one having `parallel` immediately inside `target` and the other not

* <font color="green">Exercise: change `OMP_NUM_THREADS` below and compare them with the displayed kernel launch parameters</font>

### 14-1-1. clang++
* A simple case in which `parallel` is _immediately_ inside `target`


In [ ]:
%%writefile omp_count_threads_tight.cc
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>
int getenv_int(const char * v) {
  char * s = getenv(v);
  if (!s) {
    fprintf(stderr, "specify environment variable %s\n", v);
    exit(1);
  }
  return atoi(s);
}

int main(int argc, char ** argv) {
  int H = getenv_int("OMP_NUM_THREADS");
  int num_threads_actual = 0;
#pragma omp target teams map(tofrom: num_threads_actual)
#pragma omp parallel num_threads(H)
  {
#pragma omp atomic
    num_threads_actual++;
  }
  printf("H=%d, num_threads_actual=%d\n", H, num_threads_actual);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall -fopenmp --offload-arch=sm_90 omp_count_threads_tight.cc -o omp_count_threads_tight_cl

In [ ]:
%%sub
H=32
LIBOMPTARGET_INFO=-1 OMP_TARGET_OFFLOAD=MANDATORY OMP_NUM_TEAMS=1 OMP_NUM_THREADS=${H} ./omp_count_threads_tight_cl

* A general case in which `parallel` is not immediately inside `target`


In [ ]:
%%writefile omp_count_threads_no_tight.cc
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>
int getenv_int(const char * v) {
  char * s = getenv(v);
  if (!s) {
    fprintf(stderr, "specify environment variable %s\n", v);
    exit(1);
  }
  return atoi(s);
}

int main(int argc, char ** argv) {
  int H = getenv_int("OMP_NUM_THREADS");
  int num_threads_actual = 0;
#pragma omp target teams map(tofrom: num_threads_actual)
  {
    printf("%d/%d outside parallel region\n", omp_get_team_num(), omp_get_num_teams());
#pragma omp parallel num_threads(H)
    {
#pragma omp atomic
      num_threads_actual++;
    }
  }
  printf("H=%d, num_threads_actual=%d\n", H, num_threads_actual);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall -fopenmp --offload-arch=sm_90 omp_count_threads_no_tight.cc -o omp_count_threads_no_tight_cl

In [ ]:
%%sub
H=32
LIBOMPTARGET_INFO=-1 OMP_TARGET_OFFLOAD=MANDATORY OMP_NUM_TEAMS=1 OMP_NUM_THREADS=${H} ./omp_count_threads_no_tight_cl

### 14-1-2. nvc++
* A simple case in which `parallel` is _immediately_ inside `target`


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
nvc++ -mp=gpu -Minfo=mp omp_count_threads_tight.cc -o omp_count_threads_tight_nv

In [ ]:
%%sub
H=32
NVCOMPILER_ACC_NOTIFY=1 OMP_TARGET_OFFLOAD=MANDATORY OMP_NUM_TEAMS=1 OMP_NUM_THREADS=${H} ./omp_count_threads_tight_nv 2>&1

* A general case in which `parallel` is not immediately inside `target`


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
nvc++ -mp=gpu -Minfo=mp omp_count_threads_no_tight.cc -o omp_count_threads_no_tight_nv

In [ ]:
%%sub
H=32
NVCOMPILER_ACC_NOTIFY=1 OMP_TARGET_OFFLOAD=MANDATORY OMP_NUM_TEAMS=1 OMP_NUM_THREADS=${H} ./omp_count_threads_no_tight_nv 2>&1

## 14-2. Summary
### 14-2-1. Launch parameter (thread block size)
* It is a line from the runtime system
  * clang++ : `omptarget device 0 info: Launching kernel ... with [1,1,1] blocks and [128,1,1] threads in Generic-SPMD mode`
  * nvc++ : `launch CUDA kernel file=... function=... line=... device=0 host-threadid=0 num_teams=0 thread_limit=... kernelname=... grid=<<<1,1,1>>> block=<<<1024,1,1>>> shmem=8820b`

* When `parallel` comes immediately inside `target`, so $H$ is surely known at `target`
  * clang++ : observes the value specified by `num_threads`, except it is limited by the kernel register usage or hard limit ($<=$ 1,024)
  * nvc++ : in addition to the same limitation, it somehow rounded the value specified by `num_threads` down to a nearest multiple 32 (e.g., 100 -> 96), except numbers less than 32 becomes 1 (just to avoid zero)

* When `parallel` does not immediately come inside `target`, they _resort to a predetermined default_
  * clang++ : 128, except it is limited by the kernel register usage / shared memory (if used at all)
  * nvc++ : the value determined by the kernel register usage / shared memory (if used at all)

* To display, 

|         |           | `blockDim`                            |
|---------|-----------|---------------------------------------|
| clang++ | tight     | $\min(H,\ K)$                         |
| clang++ | non-tight | $\min(128, K)$                        |
| nvc++   | tight     | $\max\bigl(32,\ \min(R(H),\ K)\bigr)$ |
| nvc++   | non-tight | $K$                                   |

where $R(H) = 32\lfloor H/32 \rfloor$ ($H$ rounded _down_ to a multiple of 32) and $K$ is whatever limit is imposed by the kernel's register usage / shared memory (if used at all), which is always $<= 1024$ (hardware limit)

* The choice of nvc++ may be bad when the user-supplied number of threads $H$ is much smaller than $K$, as it ends up running only $H$ threads, still occupying the resources assuming $K$ threads

### 14-2-2. The actual number of threads in parallel region
* It is the value shown as `num_threads_actual` of the above programs
  * clang++ simply takes minimum of $H$ and the block size; the default value used when it is not known at `target` is 128, so some slots are wasted when $H$ is smaller than 128, but such small numbers are not recommended block size anways; 
  * nvc++ somehow rounds down $H$ to a nearest multiple 32, _and subtracts 32 from it; the purpose appears to "reserve" the last 32 threads for executing regions outside `parallel` region, but the reason they do even when `parallel` comes immediately inside `target` is unclear; also, the default block size ($K$) may be as large as 1024, and waste is huge when $H$ is much smaller than 1024

|         | the actual number of CUDA threads   |
|---------|-------------------------------------|
| clang++ | $\min(H,\ \mathtt{blockDim})$      |
| nvc++   | $\max\bigl(1,\ \min(R(H),\ \mathtt{blockDim} - 32)\bigr)$ |


## 14-3. Block IDs, thread IDs, and SM number
* A CUDA thread can know its block and thread ID via `blockIdx` and `threadIx`, respectively
* From OpenMP programs, we can obtain them via assembly (PTX) code
* `omp_gpu_ptx.h` wraps them in ordinary-looking functions

 |function          |CUDA equivalent|meaning                              |
 |------------------|---------------|-------------------------------------|
 |`gpu_block_idx()` |`blockIdx.x`   |which thread block am I in           |
 |`gpu_grid_dim()`  |`gridDim.x`    |how many thread blocks were launched |
 |`gpu_thread_idx()`|`threadIdx.x`  |which thread of the block am I       |
 |`gpu_block_dim()` |`blockDim.x`   |how many threads my block really has |
 |`gpu_sm_id()`     |`%smid`        |which SM am I running on             |
 |`gpu_nsm()`       |`%nsmid`       |how many SMs this GPU has            |
 |`gpu_warp_id()`   |`%warpid`      |which warp of the block am I in      |
 |`gpu_lane_id()`   |`%laneid`      |which lane of the warp am I          |

* <font color=red>Unfortunately,</font> these work with `clang++` only
* The inline PTX must be guarded by a macro that only the device pass of the compiler defines, and `nvc++` preprocesses the file just once and defines no such macro (under `nvc++` every function returns `4294967295`)
* Getting the same information out of `nvc++` is more involved; write a genuine CUDA `__device__` functions in a `.cu` file and link them as relocatable device code
```
/* gpu_ptx.cu */
extern "C" __device__ unsigned gpu_tid(void)   { return threadIdx.x; }
extern "C" __device__ unsigned gpu_ctaid(void) { return blockIdx.x; }
```
```
/* declare them like this in the OpenMP file */
#pragma omp declare target
extern "C" unsigned gpu_tid(void);
extern "C" unsigned gpu_ctaid(void);
#pragma omp end declare target
```
```
nvcc  -arch=sm_90 -rdc=true -c gpu_ptx.cu -o gpu_ptx.o
nvc++ -mp=gpu -cuda -gpu=rdc your_program.cc gpu_ptx.o -o your_program
```
  * both `-gpu=rdc` (otherwise `ptxas fatal : Unresolved extern function`) and `-cuda` (otherwise the CUDA runtime is not linked) are necessary
  * objects are not interchangeable between the two compilers; feeding a `clang++`-compiled object to `nvc++` gives `nvlink error : Undefined reference`, because each compiler links the device images with its own tool chain


In [ ]:
%%writefile omp_gpu_ptx.h
/*** omp_gpu_ptx.h
     Read NVIDIA GPU special registers from inside an OpenMP target region.
     They let you check for yourself how OpenMP's teams and threads map onto
     CUDA's thread blocks and threads, and which SM a team ended up on.

     Usage:
       #include "omp_gpu_ptx.h"
       ...
       #pragma omp target teams num_teams(T)
       #pragma omp parallel num_threads(H)
       printf("team %d/%d thread %d/%d -> block %u/%u thread %u/%u on SM %u\n",
              omp_get_team_num(), omp_get_num_teams(),
              omp_get_thread_num(), omp_get_num_threads(),
              gpu_block_idx(), gpu_grid_dim(),
              gpu_thread_idx(), gpu_block_dim(), gpu_sm_id());

     These are only meaningful inside a target region running on an NVIDIA GPU.
     The inline PTX cannot be compiled for the host, so it is guarded by a
     macro that only the device pass of the compiler defines; outside the
     device pass the functions return 0xffffffff.

     <<< WORKS WITH clang++ ONLY >>>
     clang++ compiles the file twice, defining __NVPTX__ in the device pass,
     so the guard does the right thing.  nvc++ preprocesses the file only
     once and defines no such macro, so with nvc++ every function here
     returns 0xffffffff (4294967295).  Use clang++ for these experiments. ***/
#pragma once

#if defined(__NVPTX__) || defined(__CUDA_ARCH__) || defined(__NVCOMPILER_GPU__)
#define GPU_SREG(reg) ({ unsigned r_; asm("mov.u32 %0, %%" reg ";" : "=r"(r_)); r_; })
#else
#define GPU_SREG(reg) 0xffffffffu
#endif

#pragma omp declare target

/* blockIdx.x : which thread block am I in? */
static inline unsigned gpu_block_idx(void)  { return GPU_SREG("ctaid.x");  }
/* gridDim.x : how many thread blocks were actually launched? */
static inline unsigned gpu_grid_dim(void)   { return GPU_SREG("nctaid.x"); }
/* threadIdx.x : which thread within the block am I? */
static inline unsigned gpu_thread_idx(void) { return GPU_SREG("tid.x");    }
/* blockDim.x : how many threads does my block actually have? */
static inline unsigned gpu_block_dim(void)  { return GPU_SREG("ntid.x");   }
/* which SM (streaming multiprocessor) am I running on? */
static inline unsigned gpu_sm_id(void)      { return GPU_SREG("smid");     }
/* how many SMs does this device have? */
static inline unsigned gpu_nsm(void)        { return GPU_SREG("nsmid");    }
/* warp index within the block, lane index within the warp */
static inline unsigned gpu_warp_id(void)    { return GPU_SREG("warpid");   }
static inline unsigned gpu_lane_id(void)    { return GPU_SREG("laneid");   }

#pragma omp end declare target

In [ ]:
%%writefile omp_gpu_ptx_demo.cc
/*** Does a `team` really correspond to a CUDA thread block,
     and a `thread` to a CUDA thread?  Ask the hardware.
     Compile with clang++ (see omp_gpu_ptx.h).                       ***/
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>
#include "omp_gpu_ptx.h"

int main(int argc, char ** argv) {
  int i = 1;
  int n_teams   = (argc > i ? atoi(argv[i]) : 2); i++;
  int n_threads = (argc > i ? atoi(argv[i]) : 4); i++;
  printf("%-26s | %-28s | %s\n", "OpenMP says", "the GPU says", "where");
#pragma omp target teams num_teams(n_teams)
#pragma omp parallel num_threads(n_threads)
  printf("team %3d/%3d thread %4d/%4d | block %3u/%3u thread %4u/%4u | SM %3u/%3u warp %2u lane %2u\n",
         omp_get_team_num(), omp_get_num_teams(),
         omp_get_thread_num(), omp_get_num_threads(),
         gpu_block_idx(), gpu_grid_dim(),
         gpu_thread_idx(), gpu_block_dim(),
         gpu_sm_id(), gpu_nsm(), gpu_warp_id(), gpu_lane_id());
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall -fopenmp --offload-arch=sm_90 omp_gpu_ptx_demo.cc -o omp_gpu_ptx_demo

In [ ]:
%%sub
OMP_TARGET_OFFLOAD=MANDATORY ./omp_gpu_ptx_demo 2 4

* with `clang++` the correspondence is exact:
```
team   0/  2 thread    0/   4 | block   0/  2 thread    0/   4 | SM   2/ 32 warp  0 lane  0
team   0/  2 thread    1/   4 | block   0/  2 thread    1/   4 | SM   2/ 32 warp  0 lane  1
team   1/  2 thread    0/   4 | block   1/  2 thread    0/   4 | SM   0/ 32 warp  0 lane  0
```
  * `omp_get_team_num()` $=$ `blockIdx.x` and `omp_get_num_teams()` $=$ `gridDim.x`
  * `omp_get_thread_num()` $=$ `threadIdx.x` and `omp_get_num_threads()` $=$ `blockDim.x`
* so on this platform, <font color=blue>a team _is_ a thread block and an OpenMP
thread _is_ a CUDA thread</font>.  The spec does not promise this --- it only
speaks of teams and contention groups --- but it is what both implementations do

* <font color=green>run the program with more teams than the GPU has SMs (the MIG
partition we use has 32) and look at the `SM` column.  How are teams assigned to
SMs?  What happens to the assignment when you ask for one team per SM, and when
you ask for many more?</font>


In [ ]:
%%sub
OMP_TARGET_OFFLOAD=MANDATORY ./omp_gpu_ptx_demo 8 1